# 02 — VP-SGLD 샘플 생성

**Step 2/4**: Step 1 에서 만든 per-split ensemble 로 VP-SGLD synthetic 데이터 생성.

- VP 하이퍼파라미터를 직접 지정 (개별 셀)
- 전체 task 를 GPU 에 flattened 분배 (최대 병렬화)
- 병렬화 크기 `GPUS`, `PROCS_PER_GPU` 로 조절

## 0. Setup

In [35]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as _mp
try: _mp.set_start_method('spawn', force=True)
except RuntimeError: pass

os.chdir('/home/work/JooKyung/TabEBM')
sys.path.insert(0, 'experiments'); sys.path.insert(0, 'src')

import numpy as np

def _fmt(s):
    s = int(s); m, s = divmod(s, 60)
    return f'{m}m{s:02d}s' if m else f'{s}s'

print('ready')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
ready


## 1. eval_dir 선택 (01 에서 생성)

In [36]:
fair_root = Path('experiments/fair_eval')
if fair_root.exists():
    for p in sorted(fair_root.iterdir()):
        if p.is_dir() and (p / 'config.json').exists():
            cfg = json.loads((p / 'config.json').read_text())
            ens_ok = (p / 'ensembles' / 'split_0' / 'c0' / 'meta.json').exists()
            smp_ok = (p / 'samples').exists()
            print(f'  {p.name}  K={cfg["K"]} splits={cfg["n_splits"]}  '
                  f'ens={"OK" if ens_ok else "--"}  samples={"OK" if smp_ok else "--"}')
else:
    print('  (없음 — 01 먼저)')

  20260417_215937  K=10 splits=10  ens=OK  samples=OK


In [37]:
EVAL_DIR = Path('experiments/fair_eval/20260417_215937')   # <-- 변경!
assert (EVAL_DIR / 'config.json').exists()
assert (EVAL_DIR / 'ensembles').exists(), 'ensembles/ 없음 — 01 먼저'

config = json.loads((EVAL_DIR / 'config.json').read_text())
print(f'EVAL_DIR: {EVAL_DIR}')
print(f'  K={config["K"]}, methods={config["methods"]}')
print(f'  {config["n_splits"]} splits, {config["n_samples"]} samples, '
      f'{config["n_features"]}d, classes={config["classes"]}')

EVAL_DIR: experiments/fair_eval/20260417_215937
  K=10, methods=['Distance']
  10 splits, 100 samples, 9d, classes=[0, 1]


## 2. VP-SGLD 하이퍼파라미터

각 파라미터를 별도 셀에서 조절합니다. 주석 처리된 대안들 참고.

In [38]:
# === 2-1. beta (preconditioner 강도) ===
# VP-SGLD 기본값: 1e6 (auto_beta=True)
# 논문에 없는 파라미터 — auto_beta 로 dimensionless 스케일링
BETA = [0, 0.01, 0.1, 1, 5, 10, 100]
# BETA = [1e6, 1e8, 1e10]              # sweep
# BETA = [1e6, 1e7, 1e8, 1e9, 1e10]
# BETA = [1.0, 10.0, 100.0]            # auto_beta=False 일 때
print(f'BETA = {BETA}')

BETA = [0, 0.01, 0.1, 1, 5, 10, 100]


In [39]:
# === 2-2. eta (step size) ===
# VP-SGLD 기본값: 0.05  |  TabEBM 논문 SGLD: 0.1
# ETA = [0.05]
# ETA = [0.01, 0.05, 0.1]              # sweep
# ETA = [0.01, 0.05, 0.1, 0.2, 1.0]
ETA = [0.01, 0.1, 1]
print(f'ETA = {ETA}')

ETA = [0.01, 0.1, 1]


In [40]:
# === 2-3. tau (noise temperature) ===
# VP-SGLD 기본값: 1.0  |  TabEBM 논문: noise_std=0.01 (temperature 개념 없음)
# TAU = [1.0]
# TAU = [0.1, 0.5, 1.0, 2.0]           # sweep
# TAU = [0.1, 1.0, 10.0]
# TAU = [0.0005]
TAU = [0.0005, 0.005, 0.05, 0.5]
print(f'TAU = {TAU}')

TAU = [0.0005, 0.005, 0.05, 0.5]


In [43]:
# === 2-4. sigma_start (초기 perturbation) ===
# VP-SGLD 기본값: 0.1  |  TabEBM 논문: 0.01
SIGMA_START = [0.01, 0.1]
# SIGMA_START = [0.01, 0.1, 0.5]        # sweep
# SIGMA_START = [0.01, 0.1, 0.5, 2.0, 5.0]
print(f'SIGMA_START = {SIGMA_START}')

SIGMA_START = [0.01, 0.1]


In [44]:
# === 2-5. n_steps (SGLD 최대 step 수) ===
# VP-SGLD 기본값: 50  |  TabEBM 논문: 200
# TRAJ_CHECKPOINTS 사용 시: 여기서는 최대값만 설정,
# step 별 성능 비교는 04_evaluate 에서 N_STEPS_SWEEP 으로
N_STEPS = [200]
# N_STEPS = [50]                         # 빠르게
# N_STEPS = [200]                        # 논문 기준
# N_STEPS = [1000]                       # 긴 chain
print(f'N_STEPS = {N_STEPS}')

N_STEPS = [200]


In [45]:
# === 2-6. ignore_variance ===
# False: M = (I + beta * diag(Sigma))^{-1}  (VP-SGLD)
# True:  M = I  (plain SGLD)
IGNORE_VARIANCE = [False, True]
# IGNORE_VARIANCE = [False, True]        # 둘 다 비교
print(f'IGNORE_VARIANCE = {IGNORE_VARIANCE}')

IGNORE_VARIANCE = [False, True]


In [46]:
# === 2-7. auto_beta ===
AUTO_BETA = [False, True]
# AUTO_BETA = [True, False]             # 둘 다 비교
print(f'AUTO_BETA = {AUTO_BETA}')

AUTO_BETA = [False, True]


In [47]:
# === VP_CONFIGS 자동 조립 (모든 조합) ===
import itertools

# 이름에 모든 파라미터 포함 → 같은 이름 = 같은 파라미터 보장
# COMPACT_NAMES = True: sweep 파라미터만 이름에 (짧지만 충돌 가능)
# COMPACT_NAMES = False: 모든 파라미터 이름에 (길지만 절대 충돌 안 함)
COMPACT_NAMES = False
# COMPACT_NAMES = True

VP_CONFIGS = []
for beta, eta, tau, sigma, steps, ig_var, ab in itertools.product(
    BETA, ETA, TAU, SIGMA_START, N_STEPS, IGNORE_VARIANCE, AUTO_BETA):

    if COMPACT_NAMES:
        parts = []
        if len(BETA) > 1: parts.append(f'b{beta:g}')
        if len(ETA) > 1: parts.append(f'e{eta:g}')
        if len(TAU) > 1: parts.append(f't{tau:g}')
        if len(SIGMA_START) > 1: parts.append(f's{sigma:g}')
        if len(N_STEPS) > 1: parts.append(f'T{steps}')
        if len(IGNORE_VARIANCE) > 1: parts.append(f'iv{ig_var}')
        if len(AUTO_BETA) > 1: parts.append(f'ab{ab}')
        name = 'vp_' + ('_'.join(parts) if parts else 'default')
    else:
        name = f'vp_b{beta:g}_e{eta:g}_t{tau:g}_s{sigma:g}_T{steps}'
        if ig_var: name += '_ivT'
        if not ab: name += '_abF'

    VP_CONFIGS.append(dict(
        name=name, beta=beta, eta=eta, tau=tau,
        sigma_start=sigma, n_steps=steps,
        auto_beta=ab, ignore_variance=ig_var,
    ))

print(f'{len(VP_CONFIGS)} VP configs:')
for c in VP_CONFIGS:
    print(f'  {c["name"]}')

672 VP configs:
  vp_b0_e0.01_t0.0005_s0.01_T200_abF
  vp_b0_e0.01_t0.0005_s0.01_T200
  vp_b0_e0.01_t0.0005_s0.01_T200_ivT_abF
  vp_b0_e0.01_t0.0005_s0.01_T200_ivT
  vp_b0_e0.01_t0.0005_s0.1_T200_abF
  vp_b0_e0.01_t0.0005_s0.1_T200
  vp_b0_e0.01_t0.0005_s0.1_T200_ivT_abF
  vp_b0_e0.01_t0.0005_s0.1_T200_ivT
  vp_b0_e0.01_t0.005_s0.01_T200_abF
  vp_b0_e0.01_t0.005_s0.01_T200
  vp_b0_e0.01_t0.005_s0.01_T200_ivT_abF
  vp_b0_e0.01_t0.005_s0.01_T200_ivT
  vp_b0_e0.01_t0.005_s0.1_T200_abF
  vp_b0_e0.01_t0.005_s0.1_T200
  vp_b0_e0.01_t0.005_s0.1_T200_ivT_abF
  vp_b0_e0.01_t0.005_s0.1_T200_ivT
  vp_b0_e0.01_t0.05_s0.01_T200_abF
  vp_b0_e0.01_t0.05_s0.01_T200
  vp_b0_e0.01_t0.05_s0.01_T200_ivT_abF
  vp_b0_e0.01_t0.05_s0.01_T200_ivT
  vp_b0_e0.01_t0.05_s0.1_T200_abF
  vp_b0_e0.01_t0.05_s0.1_T200
  vp_b0_e0.01_t0.05_s0.1_T200_ivT_abF
  vp_b0_e0.01_t0.05_s0.1_T200_ivT
  vp_b0_e0.01_t0.5_s0.01_T200_abF
  vp_b0_e0.01_t0.5_s0.01_T200
  vp_b0_e0.01_t0.5_s0.01_T200_ivT_abF
  vp_b0_e0.01_t0.5_s0.01_T200_

## 3. 병렬화 + 생성 설정

In [ ]:
GPUS            = [0, 1, 2, 3]
# GPUS          = [0, 1]
# GPUS          = [0]

PROCS_PER_GPU   = 7
# PROCS_PER_GPU = 2

N_SYN_PER_CLASS = 500
# N_SYN_PER_CLASS = 500
# N_SYN_PER_CLASS = 10000         # N_SYN sweep 용

INCLUDE_SINGLE  = True
# INCLUDE_SINGLE = False

SKIP_EXISTING   = True            # True: 이미 있는 샘플 skip (기본)
# SKIP_EXISTING = False           # False: 전부 재생성 (덮어쓰기)

# === trajectory checkpoint (N_STEPS sweep 용) ===
TRAJ_CHECKPOINTS = None
# TRAJ_CHECKPOINTS = [5, 10, 20, 50, 100, 150, 200, 300, 500]

N_GPU = len(GPUS)
MAX_WORKERS = N_GPU * PROCS_PER_GPU
print(f'GPUs={GPUS}, workers={MAX_WORKERS}, N_syn={N_SYN_PER_CLASS}')
print(f'SKIP_EXISTING={SKIP_EXISTING}')

GPUs=[0, 1, 2, 3], workers=28, N_syn=250
SKIP_EXISTING=True


## 4. 샘플 생성 (병렬)

In [ ]:
from fair_eval_worker import run_one_sgld_task

data = np.load(EVAL_DIR / 'data.npz')
X_all, y_all = data['X_all'], data['y_all']
sp = np.load(EVAL_DIR / 'splits.npz')
splits = [(sp[f'tr_{i}'], sp[f'val_{i}']) for i in range(config['n_splits'])]
classes = config['classes']

ens_dir = str(EVAL_DIR / 'ensembles')
samples_dir = EVAL_DIR / 'samples'
samples_dir.mkdir(exist_ok=True)

# task 구성 — 이미 존재하는 샘플은 skip
sgld_tasks = []; skipped = 0; task_id = 0
for split_i, (tr, _val) in enumerate(splits):
    for ci, cfg in enumerate(VP_CONFIGS):
        for c in classes:
            if SKIP_EXISTING:
                out_path = samples_dir / f'split_{split_i}' / cfg['name'] / f'c{c}.npy'
                if out_path.exists():
                    skipped += 1; task_id += 1; continue
            gpu = GPUS[task_id % N_GPU]
            task_cfg = {**cfg, '_ci': ci}
            if TRAJ_CHECKPOINTS:
                task_cfg['_traj_checkpoints'] = TRAJ_CHECKPOINTS
            sgld_tasks.append(('vp', split_i, c, task_cfg,
                               tr, X_all, y_all, N_SYN_PER_CLASS,
                               config['seed'], gpu, ens_dir))
            task_id += 1
    if INCLUDE_SINGLE:
        for c in classes:
            if SKIP_EXISTING:
                out_path = samples_dir / f'split_{split_i}' / 'tabebm_single' / f'c{c}.npy'
                if out_path.exists():
                    skipped += 1; task_id += 1; continue
            gpu = GPUS[task_id % N_GPU]
            sgld_tasks.append(('single', split_i, c, config['seed'],
                               tr, X_all, y_all, N_SYN_PER_CLASS,
                               config['seed'], gpu, ens_dir))
            task_id += 1

n_total = len(sgld_tasks)
print(f'{n_total} tasks (skipped {skipped}), {MAX_WORKERS} workers')
if n_total == 0:
    print('모든 샘플 이미 존재 — 생성 건너뜀')
else:
    print()
    t0 = time.time(); done = 0
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(run_one_sgld_task, t): t for t in sgld_tasks}
        for f in as_completed(futs):
            r = f.result()
            out = samples_dir / f'split_{r["split_i"]}' / r['cfg_name']
            out.mkdir(parents=True, exist_ok=True)
            np.save(out / f'c{r["class_c"]}.npy', r['samples'])
            if r.get('traj_dict') is not None:
                np.savez_compressed(out / f'c{r["class_c"]}_traj.npz', **r['traj_dict'])
            done += 1
            if done % 10 == 0 or done == n_total:
                elapsed = time.time() - t0
                eta = elapsed / done * (n_total - done)
                print(f'  [{done:>3d}/{n_total}]  '
                      f'elapsed {_fmt(elapsed)}  ETA {_fmt(eta)}', flush=True)
    print(f'\nDone -- {_fmt(time.time()-t0)}')

## 5. 검증

In [30]:
expected = [c['name'] for c in VP_CONFIGS]
if INCLUDE_SINGLE: expected.append('tabebm_single')

ok = 0; fail = 0
for i in range(config['n_splits']):
    missing = []
    for cn in expected:
        for c in classes:
            if (samples_dir / f'split_{i}' / cn / f'c{c}.npy').exists():
                ok += 1
            else:
                missing.append(f'{cn}/c{c}'); fail += 1
    status = 'OK' if not missing else f'MISSING {missing[:3]}'
    print(f'  split_{i}: {status}')

print(f'\nTotal: {ok} OK, {fail} missing')
print(f'EVAL_DIR = {EVAL_DIR}  --> visualize / 04_evaluate 에 입력')

  split_0: OK
  split_1: OK
  split_2: OK
  split_3: OK
  split_4: OK
  split_5: OK
  split_6: OK
  split_7: OK
  split_8: OK
  split_9: OK

Total: 100 OK, 0 missing
EVAL_DIR = experiments/fair_eval/20260417_215937  --> visualize / 04_evaluate 에 입력
